In [6]:
import xarray as xr
import seaborn as sns
import numpy as np
import xarray as xr
from functools import wraps
from scipy.interpolate import interp1d
from skimage.measure import find_contours
from haversine import haversine, Unit
from skimage.morphology import convex_hull_image
from typing import List, Tuple
import functools
import time
import numpy as np
import matplotlib.pyplot as plt
import cmocean
import pickle
import json
import inspect
import LENSfunctions as funcs
import measuresfuncs as measures
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [23]:
def save_json_results(results, outdir="./IntensityMeasuresRadius2"):
    import os
    os.makedirs(outdir, exist_ok=True)

    for key, value in results.items():
        # Convert NumPy arrays to lists
        python_friendly = json.loads(json.dumps(value, default=lambda x: x.tolist() if hasattr(x, 'tolist') else x))

        filename = f"{key.replace(' ', '_').replace('/', '-')}.json"
        filepath = os.path.join(outdir, filename)

        with open(filepath, "w") as f:
            json.dump(python_friendly, f, indent=2)

        print(f"Saved → {filepath}")

In [9]:
def log_execution_time(toggle_attr='use_decorators'):
    """Decorator to log execution time, which can be toggled on/off using a class attribute."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(self, *args, **kwargs):
            if getattr(self, toggle_attr, True):  # Check if decorators are enabled
                start_time = time.time()
                result = func(self, *args, **kwargs)
                end_time = time.time()
                print(f"{func.__name__} executed in {end_time - start_time:.4f} seconds")
                return result
            else:
                return func(self, *args, **kwargs)  # Run function normally if disabled
        return wrapper
    return decorator

In [28]:
class ShapeMeasures:
    def __init__(self, lat_resolution: float = 110.574, lon_resolution: float = 111.320, use_decorators: bool = True):
        """
        A class to compute shape-based metrics for geospatial objects.

        Parameters:
            lat_resolution (float): Resolution of latitude in km per degree.
            lon_resolution (float): Resolution of longitude in km per degree at the equator.
            use_decorators (bool): Toggle for using decorators like execution time logging.
        """
        self.lat_resolution = lat_resolution
        self.lon_resolution = lon_resolution
        self.use_decorators = use_decorators  # Control decorator execution
    
    @log_execution_time()
    def calculate_area(self, lats: List[float], lons: List[float]) -> float:
        """Computes area in square kilometers."""
        y, x = np.array(lats), np.array(lons)
        dlon = np.cos(np.radians(y)) * self.lon_resolution
        dlat = self.lat_resolution * np.ones(len(dlon))
        return np.sum(dlon * dlat)

    @log_execution_time()
    def calculate_spatial_extents(self, one_obj: xr.Dataset) -> dict:
        """Computes spatial extents and summary statistics."""
        spatial_extents = []
        coords_full = []

        for i in range(len(one_obj.time)):
            stacked = one_obj[i, :, :].stack(zipcoords=['lat', 'lon'])
            intermed = stacked.dropna(dim='zipcoords').zipcoords.values

            if len(intermed) == 0:
                coords_full.append([])
                spatial_extents.append(0.0)
                continue

            lats, lons = zip(*intermed)
            coords = list(zip(lats, lons))
            coords_full.append(coords)
            spatial_extents.append(self.calculate_area(lats, lons))
            
            final_data_spatial = {
            'coords_full': coords_full,
            'spatial_extents': spatial_extents,
            'max_spatial_extent': max(spatial_extents, default=0.0),
            'max_spatial_extent_time': np.argmax(spatial_extents) if spatial_extents else -1,
            'mean_spatial_extent': np.mean(spatial_extents) if spatial_extents else 0.0,
            'cumulative_spatial_extent': np.sum(spatial_extents) if spatial_extents else 0.0,
        }
            return final_data_spatial

In [11]:
def intensity_measures(sstafilepath , mhwfilepath):

    ens_memb_ind = range(100)
    da_ssta = xr.open_mfdataset(sstafilepath, combine='nested', concat_dim='ensemble_member')
    da_mhwobj = xr.open_mfdataset(mhwfilepath, combine='nested', concat_dim='ensemble_member')
    
    ssta_mean_data_ls_all = []
    ssta_max_data_ls_all = []
    ssta_90percentile_data_ls_all = []
    ssta_90percentile_test_data_ls_all = []
    ssta_stdpertimestep_data_ls_all = []
    ssta_std_data_ls_all = []
    
    for ens_memb in ens_memb_ind:
        print('ENS MEMB:',ens_memb)
    
        ssta_mean_data_ens_ls_all = []
        ssta_max_data_ens_ls_all = []
        ssta_90percentile_ens_data_ls_all = []
        ssta_90percentile_test_ens_data_ls_all = []
        ssta_stdpertimestep_data_ens_ls_all = []
        ssta_std_data_ens_ls_all = []
        ssta_notrend = da_ssta.__xarray_dataarray_variable__[ens_memb,:,:,:].compute()
        blobs = da_mhwobj.labels[ens_memb,:,:,:].compute()
        blobs = da_mhwobj.labels.isel(radius=1).isel(ensemble_member=ens_memb).squeeze(drop=True).compute()
        unique_labels = np.unique(blobs.max(dim=('lat','lon')).data)[:-1]
    
        for object_id in unique_labels:
            print(' OBJ ID:', object_id)
    
            object_count_per_time = (blobs == object_id).sum(dim=['lat', 'lon'])
            true_time_steps = object_count_per_time.time.where(object_count_per_time > 0, drop=True)
            one_obj = blobs.sel(time=true_time_steps.time)
            one_obj_ones = xr.where(one_obj > 0, 1., 0)
            one_obj_ssta = ssta_notrend.sel(time=true_time_steps.time)
            masked_one_obj_ssta = one_obj_ssta*one_obj_ones
            masked_one_obj_ssta_nans = xr.where(masked_one_obj_ssta >0, masked_one_obj_ssta, np.nan)
    
            mean_ssta = masked_one_obj_ssta_nans.mean(dim=('lat','lon'))
            max_ssta = masked_one_obj_ssta_nans.max(dim=('lat','lon'))
            percentile_90 = masked_one_obj_ssta_nans.quantile(0.9, dim='time')
            percentile_90_per_timestep = masked_one_obj_ssta_nans.quantile(0.9, dim=('lat', 'lon'))
            ssta_std_per_timestep = masked_one_obj_ssta_nans.std(dim=('lat', 'lon'))
            ssta_std = masked_one_obj_ssta_nans.std()
    
            ssta_mean_data_ens_ls_all.append(mean_ssta.data)
            ssta_max_data_ens_ls_all.append(max_ssta.data)
            ssta_90percentile_ens_data_ls_all.append(percentile_90.data)
            ssta_90percentile_test_ens_data_ls_all.append(percentile_90_per_timestep.data)
            ssta_stdpertimestep_data_ens_ls_all.append(ssta_std_per_timestep.data)
            ssta_std_data_ens_ls_all.append(ssta_std.data)
    
        ssta_mean_data_ls_all.append(ssta_mean_data_ens_ls_all)
        ssta_max_data_ls_all.append(ssta_max_data_ens_ls_all)
        ssta_90percentile_data_ls_all.append(ssta_90percentile_ens_data_ls_all)
        ssta_90percentile_test_data_ls_all.append(ssta_90percentile_test_ens_data_ls_all)
        ssta_stdpertimestep_data_ls_all.append(ssta_stdpertimestep_data_ens_ls_all)
        ssta_std_data_ls_all.append(ssta_std_data_ens_ls_all)
    
    
    final_data_ssta = {
        "Mean SSTA": ssta_mean_data_ls_all,
        "Max SSTA": ssta_max_data_ls_all,
        "90th Percentile": ssta_90percentile_data_ls_all,
        "90th Percentile Per Timestep": ssta_90percentile_test_data_ls_all,
        "Standard Deviation per Timestep": ssta_stdpertimestep_data_ls_all,
        "Standard Deviation": ssta_std_data_ls_all,
    }

    return final_data_ssta

In [12]:
# Removing the conventional method of the linear trend
# each mfdataset is 2.04 GiB
sstafilepath = [f'/glade/work/cmendiola/data_conv_lin_trend/ens_{i}_ssta.nc' for i in range(100)]
convlin_ssta = xr.open_mfdataset(sstafilepath, combine='nested', concat_dim='ensemble_member').__xarray_dataarray_variable__

mhwpaths = [f'/glade/work/cmendiola/data_conv_lin_trend/ens_{i}_mhwobj.nc' for i in range(100)]
convlin_mhw = xr.open_mfdataset(mhwpaths, combine='nested', concat_dim='ensemble_member').labels

In [13]:
intensity_measures = intensity_measures(sstafilepath, mhwpaths)

ENS MEMB: 0
 OBJ ID: 1.0
 OBJ ID: 2.0
 OBJ ID: 3.0
 OBJ ID: 4.0
 OBJ ID: 6.0
 OBJ ID: 7.0
 OBJ ID: 8.0
 OBJ ID: 9.0
 OBJ ID: 10.0
 OBJ ID: 11.0
 OBJ ID: 12.0
 OBJ ID: 13.0
 OBJ ID: 15.0
 OBJ ID: 16.0
 OBJ ID: 17.0
 OBJ ID: 18.0
 OBJ ID: 19.0
 OBJ ID: 20.0
 OBJ ID: 21.0
 OBJ ID: 23.0
 OBJ ID: 24.0
 OBJ ID: 25.0
 OBJ ID: 26.0
 OBJ ID: 27.0
 OBJ ID: 28.0
 OBJ ID: 29.0
 OBJ ID: 30.0
 OBJ ID: 31.0
 OBJ ID: 32.0
 OBJ ID: 33.0
 OBJ ID: 34.0
 OBJ ID: 35.0
 OBJ ID: 36.0
 OBJ ID: 37.0
 OBJ ID: 38.0
 OBJ ID: 39.0
 OBJ ID: 40.0
 OBJ ID: 41.0
 OBJ ID: 42.0
 OBJ ID: 43.0
 OBJ ID: 44.0
 OBJ ID: 45.0
 OBJ ID: 46.0
 OBJ ID: 47.0
 OBJ ID: 48.0
 OBJ ID: 49.0
 OBJ ID: 50.0
 OBJ ID: 51.0
 OBJ ID: 52.0
 OBJ ID: 53.0
 OBJ ID: 54.0
 OBJ ID: 55.0
 OBJ ID: 56.0
 OBJ ID: 57.0
 OBJ ID: 58.0
 OBJ ID: 59.0
 OBJ ID: 60.0
 OBJ ID: 61.0
 OBJ ID: 62.0
 OBJ ID: 64.0
 OBJ ID: 65.0
 OBJ ID: 66.0
 OBJ ID: 67.0
 OBJ ID: 68.0
 OBJ ID: 69.0
 OBJ ID: 70.0
 OBJ ID: 71.0
 OBJ ID: 72.0
 OBJ ID: 73.0
 OBJ ID: 74.0
 OBJ ID: 75.0
 O

In [16]:
save_json_results(intensity_measures)

Saved → ./IntensityMeasures/Mean_SSTA.json
Saved → ./IntensityMeasures/Max_SSTA.json
Saved → ./IntensityMeasures/90th_Percentile.json
Saved → ./IntensityMeasures/90th_Percentile_Per_Timestep.json
Saved → ./IntensityMeasures/Standard_Deviation_per_Timestep.json
Saved → ./IntensityMeasures/Standard_Deviation.json


In [17]:
sm = measures.ShapeMeasures()

In [32]:
#############################################
# 1. EXTRACT UNIQUE OBJECT LABELS (radius=2)
#############################################

all_unique_labels_ls = []
no_items = []

for ensemble_member_id in range(100):

    # select radius 2 and ensemble once
    blob_labels = convlin_mhw.isel(
        ensemble_member=ensemble_member_id,
        radius=1
    ).compute()

    # find unique event IDs over time
    unique_labels = np.unique(
        blob_labels.max(dim=('lat','lon')).data
    )[:-1]  # drop background (0)

    all_unique_labels_ls.append(unique_labels)
    no_items.append(len(unique_labels))


#############################################
# 2. LOOP AND COMPUTE SPATIAL EXTENTS
#############################################

spatial_extent_data_ls_all = []

for ensemble_member_id in range(100):
    print('**********', ensemble_member_id)

    spatial_extent_data_ls_all_ens_memb_spec = []

    # compute blobs & anomalies once per member
    blobs = convlin_mhw.isel(
        ensemble_member=ensemble_member_id,
        radius=1
    ).compute()

    ssta_notrend = convlin_ssta.isel(
        ensemble_member=ensemble_member_id
    ).compute()

    # loop through objects in this member
    for object_id in all_unique_labels_ls[ensemble_member_id]:
        print('OBJECT ID', object_id)

        # time axis mask
        object_count_per_time = (blobs == object_id).sum(dim=['lat', 'lon'])
        true_time_steps = object_count_per_time.time.where(
            object_count_per_time > 0,
            drop=True
        )

        # grab only the times when the object exists
        one_obj = blobs.sel(time=true_time_steps.time)

        # binary mask for this object
        only_one_obj = xr.where(one_obj == object_id, 1.0, np.nan)

        ##############################
        # CALCULATE SPATIAL EXTENTS
        ##############################
        spatial_extent_data = sm.calculate_spatial_extents(
            only_one_obj
        )

        spatial_extent_data_ls_all_ens_memb_spec.append(
            spatial_extent_data['spatial_extents']
        )

    # append ensemble
    spatial_extent_data_ls_all.append(
        spatial_extent_data_ls_all_ens_memb_spec
    )


********** 0
OBJECT ID 1.0
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0000 seconds
calculate_spatial_extents executed in 0.0140 seconds
OBJECT ID 2.0
calculate_area executed in 0.0000 seconds
calculate_area executed in 0.0000 seconds
calculate_spatial_extents executed in 0.0026 seconds
OBJECT ID 3.0
calculate_area executed in 0.0000 seconds
calculate_area executed in 0.0000 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0000 seconds
calculate_area executed in 0.0000 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds
calculate_area executed in 0.0001 seconds